In [1]:
import numpy as np
import torch
from torch.optim import Optimizer, Adam
import collections
from typing import Iterable, Tuple, Any, Optional, Callable

In [2]:
import sys
sys.path.append('..')
sys.path.append('.')
from dataset import ptb
from common.util import create_contexts_target, most_similar, analogy

## 对单个单词的采样的类

In [3]:
class UnigramSampler:
    """单个单词的采样
    """

    def __init__(self, corpus:list, power=0.75, sample_size:int=2):
        r"""对 corpus 进行概率分布的统计，
            并且对每个概率进行 power 次方的处理，
            以增加低频条目被抽中的概率
            公式为

            $$P'(w_i) = \frac{P(w_i)^{0.75}}{\sum_{j}^{n} P(w_j)^{0.75}}$$

        Args:
            corpus (list): 数据列表。
            power (float, optional): 每个概率进行次方 Defaults to 0.75.
            sample_size (int, optional): _description_. Defaults to 2.
        """

        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        # 统计每个条目的数量。
        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        # 得到每个条目的原始概率
        self.word_p = torch.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        # 计算每个条目的修正概率
        self.word_p = self.word_p ** power
        self.word_p /= torch.sum(self.word_p)

    def get_negative_sample(self, target:torch.Tensor) -> torch.Tensor:
        """根据目标列表对特征进行负采样

        Args:
            target (torch.Tensor): 目标列表

        Returns:
            torch.Tensor: 根据目标列表返回的负采样矩阵，形状为(batch_size, sample_size)
        """
        batch_size = target.shape[0]
        negative_sample = torch.multinomial(self.word_p.expand(batch_size, -1), self.sample_size, replacement=True)

        return negative_sample

## Embedding 层

In [4]:
class EmbeddingFunction(torch.autograd.Function):
    """自定义的 Embedding 算子，包含前向和反向传播的底层数学逻辑"""

    @staticmethod
    def forward(ctx: Any, W: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """实现 Embedding 层的正向传播，从权重矩阵中提取特定单词对应的行向量

        Args:
            ctx (Any): 上下文对象，用于存储反向传播需要的信息
            W (torch.Tensor): 权重矩阵，形状为 (V, H)，V为词汇量，H为隐藏层大小
            idx (torch.Tensor): 单词的索引张量

        Returns:
            torch.Tensor: 提取出的对应的分布式表示(词向量)
        """
        # 保存反向传播需要的信息
        ctx.save_for_backward(idx)
        ctx.W_shape = W.shape

        # 实现提取逻辑，并返回结果
        return W[idx]

    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, None]:
        """实现 Embedding 层的反向传播

        Args:
            ctx (Any): 上下文对象，包含前向传播时保存的 idx 和 W_shape
            grad_output (torch.Tensor): 后面层传回来的梯度，形状与 forward 的输出一致

        Returns:
            Tuple[torch.Tensor, None]:
            - grad_W: 对权重矩阵 W 的梯度
            - None: idx 是离散的索引，不需要梯度，返回 None
        """
        # 反向传播，重复的idx值需要相加
        idx = ctx.saved_tensors[0].to(grad_output.device)

        dW = torch.zeros(ctx.W_shape, device=grad_output.device)
        dW.index_add_(dim=0, index=idx, source=grad_output)

        return dW, None

class Embedding(torch.nn.Module):
    """Embedding 网络层，用于将单词 ID 批量转换为对应的分布式表示"""

    def __init__(self, W: torch.Tensor):
        """初始化 Embedding 层

        Args:
            W (torch.Tensor): 初始化的权重矩阵，形状为 (V, H)
        """
        super().__init__()
        # 将传入的张量注册为模型的参数，使其可以被优化器更新
        self.W = W

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """Embedding 层的前向传播调用
           这里主要为了演示retain_grad()如何截获梯度用于debug

        Args:
            idx (torch.Tensor): 输入的单词 ID 张量

        Returns:
            torch.Tensor: 对应的词向量张量
        """
        # 1. 先计算出结果
        out = EmbeddingFunction.apply(self.W, idx)
        
        # 2. 打上保留梯度的标记
        out.retain_grad()
        
        # 3. 把它存入实例属性，方便外部偷看
        self.debug_out = out

        # 最终调用打印 model.in_layer.debug_out
        
        return out

## Embedding 和内积运算合并层

In [5]:
class BatchDotFunction(torch.autograd.Function):
    """处理 h 和 target_W 批量内积的自定义算子"""

    @staticmethod
    def forward(ctx: Any, h: torch.Tensor, target_W: torch.Tensor) -> torch.Tensor:
        """
        Args:
            h (torch.Tensor): 中间层神经元张量，形状为 (batch_size, hidden_size)
            target_W (torch.Tensor): 提取出的词向量张量，形状为 (batch_size, hidden_size)

        Returns:
            torch.Tensor: 内积得分，形状为 (batch_size,)
        """
        # 1. 保存 h 和 target_W 供反向传播使用
        ctx.save_for_backward(h, target_W)

        # 2. 正向传播：计算对应元素乘积，并按行求和 (对应书中 np.sum(target_W * h, axis=1))
        return torch.sum(target_W * h, dim=1)


    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            grad_output (torch.Tensor): 从 SigmoidWithLoss 传回的梯度，形状 (batch_size,)

        Returns:
            Tuple[torch.Tensor, torch.Tensor]:
            分别对应 h 和 target_W 的梯度(返回值的形状均为(batch_size, hidden_size))
        """
        # 1. 取出保存的 h 和 target_W
        h, target_W = ctx.saved_tensors

        # 2. 请在此处实现反向传播：重塑 grad_output 并分别计算 grad_h 和 grad_target_W
        grad_h = grad_target_W = None
        grad_output = grad_output.view(-1, 1)

        if ctx.needs_input_grad[0]:
            grad_h = grad_output * target_W

        if ctx.needs_input_grad[1]:
            grad_target_W = grad_output * h

        return grad_h, grad_target_W

class EmbeddingDot(torch.nn.Module):
    """将 Embedding 层和内积运算合并的层"""

    def __init__(self, W: torch.Tensor):
        """初始化 EmbeddingDot 层

        Args:
            W (torch.Tensor): 初始化的输出侧权重矩阵，形状为 (V, H)
        """
        super().__init__()
        # 复用刚才的 Embedding 层！
        self.embed = Embedding(W)

    def forward(self, h: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """EmbeddingDot 层的前向传播调用

        Args:
            h (torch.Tensor): 中间层神经元张量，形状为 (batch_size, H)
            idx (torch.Tensor): 目标词的索引张量，形状为 (batch_size,)

        Returns:
            torch.Tensor: 对应目标词的内积得分
        """

        target_W = self.embed(idx)
        return BatchDotFunction.apply(h, target_W)

## 交叉熵与 Sigmoid 的组合层

In [6]:
class SigmoidWithLossFunction(torch.autograd.Function):
    """包含 Sigmoid 激活与二分类交叉熵损失反向推导逻辑的自定义算子"""

    @staticmethod
    def forward(ctx: Any, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        r"""实现 Sigmoid 激活和交叉熵损失的计算。

        即两个公式的实现：
        $$
        y=\frac{1}{1+e^{-x}}
        $$
        $$L = - \left[ t \cdot \log(y) + (1-t) \cdot \log(1 - y) \right]$$

        其中 t 为真实结果，y 为预测结果。

        Args:
            ctx (Any): 上下文对象，用于存储反向传播所需数据。
            x (torch.Tensor): 预测得分，形状为 (batch_size,)。
            target (torch.Tensor): 正确解标签 (0 或 1)，形状为 (batch_size,)。

        Returns:
            torch.Tensor: 计算得到的标量总损失。
        """

        # 为了防止 log(0) 导致数值不稳定，请在 y 内部加上一个极小数（如 1e-7）
        y = 1 / (1 + torch.exp(-x))
        loss = torch.mean(-(target * torch.log(y + 1e-7) + (1 - target) * torch.log(1 - y + 1e-7)))

        # 保存 y 和 target 供反向传播使用
        ctx.save_for_backward(y, target)

        return loss

    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, None]:
        """实现交叉熵与 Sigmoid 组合后的极简反向传播。

        Args:
            ctx (Any): 上下文对象，包含前向传播时保存的 y 和 target。
            grad_output (torch.Tensor): 标量损失传回的梯度，通常为 1.0 (张量)。

        Returns:
            Tuple[torch.Tensor, None]:
            - grad_x: 对输入得分 x 的梯度。
            - None: target 是真实的离散标签，不需要求导，返回 None。
        """
        # 1. 取出前向传播保存的张量
        y, target = ctx.saved_tensors

        # 2. 请在此处根据书上推导的公式 (y - t) 计算梯度
        # 别忘了乘上链式法则传进来的 grad_output
        batch_size = y.shape[0]
        
        return grad_output * (y - target) / batch_size, None

class SigmoidWithLoss(torch.nn.Module):
    """Sigmoid 与二分类交叉熵损失函数的组合网络层"""

    def __init__(self):
        """初始化 SigmoidWithLoss 层"""
        super().__init__()

    def forward(self, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """SigmoidWithLoss 层的前向传播调用

        Args:
            x (torch.Tensor): 预测得分，形状为 (batch_size,)。
            target (torch.Tensor): 正确解标签 (0 或 1)，形状为 (batch_size,)。

        Returns:
            torch.Tensor: 计算出的标量总损失。
        """
        return SigmoidWithLossFunction.apply(x, target)

## 负采样损失层

In [7]:
class NegativeSamplingLoss(torch.nn.Module):
    """负采样损失层，统合正例与负例的二分类损失。

    通过 UnigramSampler 根据语料库频率抽取负例，并利用 EmbeddingDot 计算内积得分，
    最后结合 SigmoidWithLoss 计算正例 (标签为 1) 和负例 (标签为 0) 的交叉熵损失总和。
    """

    def __init__(self, W: torch.Tensor, corpus: list, power: float = 0.75, sample_size: int = 5):
        """初始化负采样损失层。

        Args:
            W (torch.Tensor): 输出侧权重矩阵，形状为 (vocab_size, hidden_size)。
            corpus (list): 语料库单词 ID 列表，用于负采样概率分布的统计。
            power (float, optional): 概率分布的次方修正值，用于提升低频词概率。Defaults to 0.75。
            sample_size (int, optional): 为每个正例采样的负例数量。Defaults to 5。
        """
        super().__init__()
        self.sample_size = sample_size
        
        # 调用 get_negative_sample(target) 会返回形状为 (batch_size, sample_size) 的负例张量
        self.sampler = UnigramSampler(corpus, power, sample_size)

        # 实例化一个 EmbeddingDot，它会自动复用底层的 W
        self.embed_dot = EmbeddingDot(W)

        # 实例化二分类交叉熵损失层
        self.loss_fn = SigmoidWithLoss()

    def forward(self, h: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """NegativeSamplingLoss 层的前向传播调用。

        Args:
            h (torch.Tensor): 中间层神经元张量，形状为 (batch_size, hidden_size)。
            target (torch.Tensor): 目标单词 (正例) ID，形状为 (batch_size,)。

        Returns:
            torch.Tensor: 正例和所有负例计算得出的标量总损失。
        """
        # 1. 采样负例，形状为 (batch_size, sample_size)
        negative_sample = self.sampler.get_negative_sample(target)

        # --- 正例的损失计算 ---
        positive_score = self.embed_dot(h, target)
        positive_target = torch.ones_like(positive_score)
        total_loss = self.loss_fn(positive_score, positive_target)

        # --- 负例的损失计算 ---
        negative_target = torch.zeros_like(positive_score)
        for i in range(self.sample_size):
             # 提取第 i 列，形状变为 (batch_size,)
            negative_sample_idx = negative_sample[:, i]
            negative_score = self.embed_dot(h, negative_sample_idx)
            total_loss += self.loss_fn(negative_score, negative_target)
            
        return total_loss

## 后半部分的玩具数据测试

In [8]:
# ==========================================
# 1. 准备玩具数据
# ==========================================
vocab_size = 7
hidden_size = 3
batch_size = 3
sample_size = 2

# 模拟一个极小的语料库
corpus = [0, 1, 2, 3, 4, 1, 2, 3]

# 模拟输出侧权重矩阵 W_out，形状 (7, 3)
W_out = torch.arange(21, dtype=torch.float32)\
        .reshape(vocab_size, hidden_size)
# 原地将其标记为需要计算梯度的叶子节点 🍃
W_out.requires_grad_()

# 模拟中间层传过来的神经元特征 h，形状 (3, 3)
# 注意：我们设置 requires_grad=True，以便一会测试反向传播！
h = torch.tensor([
    [ 0.5,  0.1, -0.2],
    [-0.3,  0.8,  0.0],
    [ 0.0, -0.1,  0.6]
], requires_grad=True)

# 模拟这一批次的正确目标词 (正例)
target = torch.tensor([1, 3, 0])

# ==========================================
# 2. 实例化网络层并运行
# ==========================================
print("🚀 开始测试 NegativeSamplingLoss...")

# 实例化负采样损失层 (假设采样 2 个负例)
loss_layer = NegativeSamplingLoss(W_out, corpus, power=0.75, sample_size=sample_size)

# 【测试正向传播】
loss = loss_layer(h, target)
print(f"\n✅ 前向传播成功！算出的总损失 Loss: {loss.item():.4f}")

# 【测试反向传播】
loss.backward()
print("\n✅ 反向传播成功！")
print(f"👉 中间层 h 收到的梯度 (形状应为 {h.shape}):\n{h.grad}")
print(f"👉 权重 W_out 收到的梯度 (形状应为 {W_out.shape}):\n{loss_layer.embed_dot.embed.W.grad}")

🚀 开始测试 NegativeSamplingLoss...

✅ 前向传播成功！算出的总损失 Loss: 6.1384

✅ 反向传播成功！
👉 中间层 h 收到的梯度 (形状应为 torch.Size([3, 3])):
tensor([[3.3152, 3.7773, 4.2394],
        [3.8791, 4.5121, 5.1450],
        [1.9674, 2.4621, 2.9569]])
👉 权重 W_out 收到的梯度 (形状应为 torch.Size([7, 3])):
tensor([[ 0.0000, -0.0167,  0.1001],
        [-0.0206,  0.2564, -0.0281],
        [ 0.0000, -0.0328,  0.1967],
        [ 0.0617,  0.2962, -0.0643],
        [ 0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000]])


## 连续词袋模型的实现

In [9]:
class CBOW(torch.nn.Module):
    """连续词袋模型 (Continuous Bag-of-Words) 的 PyTorch 实现"""

    def __init__(self, vocab_size: int, hidden_size: int, window_size: int, corpus: list):
        """初始化 CBOW 模型。

        Args:
            vocab_size (int): 词汇表的大小 (V)。
            hidden_size (int): 中间层神经元的数量 (H)。
            window_size (int): 上下文窗口大小 (目标词左右各 window_size 个词)。
            corpus (list): 语料库单词 ID 列表，用于负采样分布统计。
        """
        super().__init__()
        self.window_size = window_size

        # 初始化输入侧和输出侧的权重矩阵，并用 nn.Parameter 包装使其可学习
        W_in = 0.01 * torch.randn(vocab_size, hidden_size, dtype=torch.float32)
        W_out = 0.01 * torch.randn(vocab_size, hidden_size, dtype=torch.float32)
        
        self.W_in = torch.nn.Parameter(W_in)
        self.W_out = torch.nn.Parameter(W_out)

        # 实例化核心网络层
        # 💡 在 PyTorch 中，我们只需要一个 Embedding 层来处理所有的输入
        self.in_layer = Embedding(self.W_in)
        self.ns_loss = NegativeSamplingLoss(self.W_out, corpus, power=0.75, sample_size=5)

    def forward(self, contexts: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """CBOW 模型的前向传播调用。

        Args:
            contexts (torch.Tensor): 上下文单词 ID，形状为 (batch_size, 2 * window_size)。
            target (torch.Tensor): 目标单词 (正例) ID，形状为 (batch_size,)。

        Returns:
            torch.Tensor: 当前批次计算出的标量总损失。
        """
        h = 0
        context_count = 2 * self.window_size
        
        # 1. 请在此处实现上下文的词向量提取与求平均,相当于特征的平均
        # 写一个 range(context_count) 的循环，
        for i in range(context_count):
            # 每次提取 contexts 的第 i 列传入 self.in_layer，累加到 h。
            h += self.in_layer(contexts[:, i])
        
        # 最后别忘了除以 context_count。
        h = h / context_count
        
        # 打标记
        h.retain_grad()
        
        # 存起来，调试用 h.grad 即可
        self.debug_h = h

        # 计算最终的损失
        # 将求均值后的 h 和 target 传入 self.ns_loss 进行计算。
        loss = self.ns_loss(h, target)
        
        return loss

In [10]:
# ==========================================
# 1. 准备 PyTorch 版玩具数据
# ==========================================
vocab_size = 7
hidden_size = 3
window_size = 1
corpus = [0, 1, 2, 3, 4, 1, 2, 3]

# contexts 形状为 (batch_size, 2 * window_size)，这里是 (3, 2)
contexts = torch.tensor([
    [0, 2],
    [1, 3],
    [2, 4]
])
# target 形状为 (batch_size,)，这里是 (3,)
target = torch.tensor([1, 2, 3])

# ==========================================
# 2. 实例化并测试
# ==========================================
print("🚀 正在初始化 CBOW 模型...")
# 为了防止样本太少报错，这里设置 sample_size=2
model = CBOW(vocab_size, hidden_size, window_size, corpus)
model.ns_loss.sample_size = 2 

print("\n🚀 开始测试前向传播与反向传播...")
loss = model(contexts, target) # 习惯上直接调用 model() 触发 forward
print(f"✅ 前向传播成功！Loss: {loss.item():.4f}")

loss.backward()
print("✅ 反向传播成功！")
print(f"👉 W_in 收到的梯度形状: {model.W_in.grad.shape}")

🚀 正在初始化 CBOW 模型...

🚀 开始测试前向传播与反向传播...
✅ 前向传播成功！Loss: 2.0794
✅ 反向传播成功！
👉 W_in 收到的梯度形状: torch.Size([7, 3])


## SDG

In [11]:
class StandardSGD(Optimizer):
    """标准的随机梯度下降 (SGD) 优化器的 PyTorch 实现。

    继承自 torch.optim.Optimizer，支持参数分组机制 (param_groups)，
    并完全符合 PyTorch 底层的优化器开发规范。
    """

    def __init__(self, params: Iterable[torch.Tensor], lr: float = 0.01):
        """初始化 StandardSGD 优化器。

        Args:
            params (Iterable[torch.Tensor]): 待优化的模型参数迭代器 
                (通常传入 model.parameters())。
            lr (float, optional): 学习率 (Learning Rate)，控制参数更新的步长。
                Defaults to 0.01.
        
        Raises:
            ValueError: 如果传入的学习率 lr 小于 0。
        """
        if lr < 0.0:
            raise ValueError(f"无效的学习率: {lr}")
            
        # 1. 定义默认超参数字典
        defaults = dict(lr=lr)
        
        # 2. 调用父类初始化，PyTorch 会自动帮我们把 params 和 defaults 组装成 self.param_groups
        super().__init__(params, defaults)


    @torch.no_grad()
    def step(self, closure: Optional[Callable[[], float]] = None) -> Optional[float]:
        """执行单步参数更新。

        Args:
            closure (Callable[[], float], optional): 一个重新评估模型并返回损失的闭包。
                对于基础 SGD 通常不需要，但在某些高级优化器 (如 L-BFGS) 中是必需的。
                Defaults to None。

        Returns:
            Optional[float]: 如果传入了 closure，则返回计算出的损失值，否则返回 None。
        """
        loss = None

        """
        closure (闭包)： 
        深度学习里绝大多数情况用不到它。
        但某些极其复杂的优化算法（比如 L-BFGS）在走一步之前，需要反复多次计算 Loss 来探路。
        这个 closure 就是用来重新计算 Loss 的函数。
        对于 SGD 来说，这只是一段为了满足 PyTorch 接口规范的“样板代码”，运行时直接忽略就好。
        """
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # 3. 遍历所有的参数组
        """
        param_groups(参数组)： 
        很多时候，我们会希望网络的不同部分用不同的学习率
        （比如预训练好的底层特征提取器学习率设得很小，而我们新加的输出层学习率设得很大）。
        PyTorch 的优化器通过将参数打包成不同的“组（groups）”来管理这种需求。
        这里的循环 for group in self.param_groups: 就是在遍历这些拥有不同设置的参数大包。
        """
        for group in self.param_groups:
            lr = group['lr']
            
            # 4. 遍历当前组内的每一个参数
            """
            group['params'] (组内参数)： 
            进入一个参数组后，内层的 for p in group['params']: 
            就是在逐个遍历这个组里包含的具体权重矩阵（比如我们 CBOW 模型里的 W_in 和 W_out）。
            """
            for p in group['params']:
                # 如果该参数没有梯度 (例如冻结了该层)，则跳过
                if p.grad is not None:
                    grad = p.grad
                    
                    # 5. 核心更新公式：W_new = W_old - lr * grad
                    # 使用原地减法 (sub_) 直接在内存中修改参数值，避免创建多余的张量拷贝
                    p.sub_(grad, alpha=lr)

        return loss
    

# ==========================================
# 2. 极简玩具测试：优化单个参数 W
# ==========================================
# 假设我们有一个参数 W，初始值为 10.0
W = torch.tensor([10.0], requires_grad=True)

# 实例化我们的优化器，把 W 交给它管理，设定学习率为 0.1
optimizer = StandardSGD([W], lr=0.1)

print(f"🔄 更新前 W 的值: {W.item():.2f}")

# 假设我们经过前向传播，算出了一个 Loss
# 为了演示，我们直接定义一个假想的损失函数 Loss = W^2
loss = W ** 2  

# 反向传播求导数 (根据微积分，导数应该是 2 * W = 2 * 10 = 20)
loss.backward()

print(f"📉 反向传播后算出的梯度 W.grad: {W.grad.item():.2f}")

# 见证奇迹的时刻：让优化器走一步！
optimizer.step()

print(f"✅ 更新后 W 的值: {W.item():.2f}")

🔄 更新前 W 的值: 10.00
📉 反向传播后算出的梯度 W.grad: 20.00
✅ 更新后 W 的值: 8.00


## 正式训练的初始化参数

- `corpus`： 是一个一维的列表，它全部由数值组成，假设text 为 `You say hello and I say goodbye.`。那么 `corpus` 的值为
$$\begin{bmatrix}
0 & 1 & 2 & 3 & 4 & 1 & 5 & 6
\end{bmatrix}$$
- `contexts`： 它可以直接理解为目标的特征，假设text 为 'You say hello and I say goodbye.'，`window_size`为1那么它的值为 
$$\begin{bmatrix}
0 & 2 \\
1 & 3 \\
2 & 4 \\
3 & 1 \\
4 & 5 \\
1 & 6
\end{bmatrix}$$
- `target`： 需要预测的单词的id,同时也是`corpus`去掉收尾`window_size`之后的值, 假设text 为 'You say hello and I say goodbye.'， `window_size`为1，那么它的值为$$\begin{bmatrix}
1 & 2 & 3 & 4 & 1 & 5
\end{bmatrix}$$
- `window_size`： 决定左右取几个单词的参数，即滑动窗口

In [12]:
hidden_size = 100
window_size = 5

corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)

contexts, target = create_contexts_target(corpus, window_size)

contexts = torch.Tensor(contexts).int()
target = torch.Tensor(target).int()

## 单次训练

下面是单次训练的整个流程

1. 定义模型
2. 定义优化器
3. 前向传播，计算损失(仅仅是0~100行的小批量数据)
4. 梯度清零
5. 反向传播
6. 优化器调整参数

In [13]:
model = CBOW(vocab_size, hidden_size, window_size, corpus)
# StandardSGD([model.W_in, model.W_out], lr=0.01)
optimizer = StandardSGD(model.parameters(), lr=0.01)

loss = model(contexts[0:100], target[0:100])
model.zero_grad()
loss.backward()
optimizer.step()

In [14]:
print(model.in_layer.debug_out)
print(h.grad)

tensor([[ 0.0123,  0.0049, -0.0043,  ...,  0.0090,  0.0104, -0.0067],
        [-0.0013, -0.0007,  0.0151,  ..., -0.0303,  0.0205, -0.0058],
        [ 0.0109, -0.0015,  0.0044,  ...,  0.0202, -0.0017, -0.0089],
        ...,
        [ 0.0018, -0.0088, -0.0136,  ...,  0.0087,  0.0020,  0.0163],
        [ 0.0160, -0.0227,  0.0029,  ..., -0.0043,  0.0024,  0.0061],
        [ 0.0177, -0.0172, -0.0049,  ..., -0.0069, -0.0067, -0.0122]],
       grad_fn=<EmbeddingFunctionBackward>)
tensor([[3.3152, 3.7773, 4.2394],
        [3.8791, 4.5121, 5.1450],
        [1.9674, 2.4621, 2.9569]])


## 真正的训练?

并不是，为了偷懒 epoch 只做了个硬性的for循环

In [15]:
batch_size = 100
steps = int(len(contexts) / batch_size)

# --- 1. 准备阶段（循环外，只执行一次） ---
# 检测 GPU 并在有条件时使用它
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 确保当前批次的数据也在相同的设备上
contexts = contexts.to(device)
target = target.to(device)

# 实例化模型并将其推送到指定设备
model = CBOW(vocab_size, hidden_size, window_size, corpus).to(device)

# 实例化优化器，把模型参数交由它管理
# optimizer = StandardSGD(model.parameters(), lr=0.01)

# 将原来的 StandardSGD 替换掉
optimizer = Adam(model.parameters(), lr=0.001) # Adam 通常使用较小的初始学习率

# --- 2. 训练阶段（假设 epochs 是总轮数，dataloader 提供批次数据） ---
for epoch in range(10):
    for step in range(steps):
        
        # 滑动窗口的大小定义
        start = step * batch_size
        end = start + batch_size

        # 1. 前向传播：计算当前批次的损失
        loss = model(contexts[start:end], target[start:end])
        
        # 2. 清空梯度：把上一轮的梯度清零 🧹
        optimizer.zero_grad()

        # 3. 反向传播：计算当前轮次的新梯度 📉
        loss.backward()
        
        # 4. 更新参数：踩下油门，让优化器走一步！ 🚀
        optimizer.step()

        if step % 1000 == 0 or step == steps - 1:
            print(f"epoch: {epoch} - step: {step}")
            print(loss)

epoch: 0 - step: 0
tensor(4.1589, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 1000
tensor(2.4499, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 2000
tensor(2.6041, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 3000
tensor(2.6582, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 4000
tensor(2.4472, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 5000
tensor(2.3001, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 6000
tensor(2.4055, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 7000
tensor(2.3820, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 8000
tensor(1.9288, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 9000
tensor(2.5155, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 0 - step: 9294
tensor(2.2031, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 1 - step: 0
tensor(2.7571, device='cuda:0', grad_fn=<AddBackward0>)
epoch: 1 - step: 1000
tensor(2.1196, device='cuda:0', grad_fn=<AddBack

## 结果权重确认

这里对训练结果权重的相似单词进行确认

In [16]:
querys = ['you', 'year', 'car', 'toyota', 'run']
word_vecs = model.W_in.to('cpu').detach().numpy()
for query in querys:
    most_similar(query, word_to_id, id_to_word, word_vecs, top=5)


[query] you
 we: 0.7619768381118774
 i: 0.719018816947937
 your: 0.6349183320999146
 reasonable: 0.6076351404190063
 anything: 0.6039077639579773

[query] year
 month: 0.8575659394264221
 week: 0.7731285095214844
 summer: 0.7643606066703796
 spring: 0.7315505146980286
 decade: 0.7080960273742676

[query] car
 truck: 0.637301504611969
 cars: 0.5963847041130066
 auto: 0.5843051671981812
 vehicles: 0.5654853582382202
 luxury: 0.5593286752700806

[query] toyota
 engines: 0.6702311635017395
 kellogg: 0.6414833664894104
 nec: 0.6412599682807922
 nissan: 0.64082932472229
 chevrolet: 0.6300204992294312

[query] run
 haul: 0.6697294116020203
 tisch: 0.5867146253585815
 dial: 0.5821848511695862
 seasons: 0.5666633248329163
 fly: 0.5527753233909607


## 单词计算

analogy 函数主要是将三个单词按照下面公式进行计算后得到的相似的单词
$$v_{\text{target}} = v_{\text{b}} - v_{\text{a}} + v_{\text{c}}$$

In [17]:
analogy('man', 'king', 'woman', word_to_id, id_to_word, word_vecs)
analogy('japan', 'toyota', 'america', word_to_id, id_to_word, word_vecs)
analogy('take', 'took', 'go', word_to_id, id_to_word, word_vecs)
analogy('good', 'better', 'bad', word_to_id, id_to_word, word_vecs)


[analogy] man:king = woman:?
 pages: 3.1019930839538574
 developed: 2.8751091957092285
 maker: 2.7620139122009277
 wife: 2.6579442024230957
 university: 2.573939561843872

[analogy] japan:toyota = america:?
 daffynition: 3.5469985008239746
 maker: 3.3724958896636963
 communications: 3.2432737350463867
 revenue: 3.1806416511535645
 co: 3.1452178955078125

[analogy] take:took = go:?
 're: 4.285768032073975
 went: 4.006138801574707
 came: 3.6617894172668457
 began: 3.6447906494140625
 was: 3.6229195594787598

[analogy] good:better = bad:?
 more: 5.352041721343994
 less: 5.309734344482422
 rather: 4.948886394500732
 greater: 3.9850172996520996
 fewer: 3.551692247390747
